# AllLife Bank

## Business Context
AllLife Bank is a mid-sized, fast-growing financial institution that offers a range of retail banking services, including savings and checking accounts, fixed deposits, and personal loans. The bank’s business model is centered on building long-term customer relationships, expanding its retail footprint, and growing its loan portfolio to drive sustainable profitability through interest income. It currently relies on a large base of liability customers (depositors) but faces a significant under-representation of asset customers (borrowers). To drive profitability through interest income, the bank must aggressively expand its loan portfolio by converting existing depositors into personal loan customers. The bank aims to optimize its personal loan marketing strategy. Rather than targeting customers based only on static attributes, the bank wants to first understand customer spending behavior and then use that behavioral insight to predict loan acceptance likelihood

### Objective
To clean, explore and visualise the dataset to:
- Predict a customers expected monthly credit card spending.
- Use that prediction with other variables to identify customers who are most likely to accept a loan offer.

### Dataset given

I have been provided with the Loan_Modelling dataset. The columns and defintions are as follows:
* `ID`: Customer ID 

* `Age`: Customer’s age in completed years 

* `Experience`: #years of professional experience 

* `Income`: Annual income of the customer (in thousand dollars) 

* `ZIP Code`: Home Address ZIP code. 

* `Family`: the Family size of the customer 

* `CCAvg`: Average spending on credit cards per month (in thousand dollars) 

* `Education`: Education Level. 1: Undergrad; 2: Graduate;3: Advanced/Professional 

* `Mortgage`: Value of house mortgage if any. (in thousand dollars) 

* `Personal_Loan`: Did this customer accept the personal loan offered in the last campaign? (0: No, 1: Yes) 

* `Securities_Account`: Does the customer have securities account with the bank? (0: No, 1: Yes) 

* `CD_Account`: Does the customer have a certificate of deposit (CD) account with the bank? (0: No, 1: Yes) 

* `Online`: Do customers use internet banking facilities? (0: No, 1: Yes) 

* `CreditCard`: Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes) 

### Importing libraries

In [ ]:
# Libraries that help with data maniuplation
import numpy as np
import pandas as pd

# Libraries that help with data visualisation
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import(mean_absolute_error,root_mean_squared_error,r2_score,mean_absolute_percentage_error)

sns.set(style = 'whitegrid')

In [ ]:
# Load the dataset
data = pd.read_csv('Loan_Modelling.csv')

In [ ]:
# Create a new copy of the data so we don't lose the original.
df = data.copy()

In [ ]:
# View the first 10 rows to get an idea of the data.
df.head(10)

Looking at the columns and definitions, the column ID is just a column that will uniquely identify the customer in the dataset and will not help us to predict if the customers average spending on a credit card or if they will accept a loan. For that reason I will drop this from our dataset.

In [ ]:
# Remove ID from our dataset.
df.drop(columns = 'ID',inplace = True)

## Dataset Overview
- How many rows/columns in our dataset?

In [ ]:
# Number of rows and columns in our dataset.
df.shape

In [ ]:
# Column names
df.columns

We already know these are the columns and the definitions of these columns are shown above.

In [ ]:
#Check the datatypes and the number of non-nulls in our dataset.
df.info()

In [ ]:
# Check for duplicates.
df.duplicated().sum()

There are no duplicates in the dataset so do not need to worry about removing them.

We see that all the variables are represented numerically, even though some are discrete categorical and others are continous.
Numerical Continous features are:
- Age
- Experience
- Income
- CCAvg(Target variable for model 1)
- Mortgage

Numerical Discrete features are:
- ZipCode
- Family
- Education
- Personal_Loan(Target variable for model 2)
- Securities_Account
- CD_Account
- Online
- CreditCard

Out of these variables for the first problem, we want to understand Customer Spending Behaviour. The target variable is CCAvg.Since CCAvg is continous, we are solving a regression problem.


We see that there are no null values in our dataset. When looking through our data-types and comparing it to the defintion above, we see that these data-types all make sense given the context. Now that we are happy with the data-types and there are no nulls, we can next look at the summary statistics to see if there are any annomalies in our data.

In [ ]:
# Check number of unique values for ZIPCode since it's categorical.
df['ZIPCode'].nunique()

In [ ]:
df.drop(columns= 'ZIPCode',inplace = True)

There are a lot of values for this field. If we were to one hot encode this in our model this would cause problems since we would have 466 extra features for this one field and that would overcomplicate the problem and for each row there would be 466 columns of 0 and one row with 1 which would make it difficult for the Machine Learning model to identify patterns. I am not sure how to represent this in my code so I will drop this from my model.

In [ ]:
# Look at the summary statistics of our numerical columns.
df.describe()

After looking at this, we see the following potential flags in the data.
- Experience should be greater or equal to 0 but we have a value of -3. We will need to investigate this.
- The minimum for CCAvg is 0. This indicates that some customers do not spend anything on their credit cards. This is not nesccarily incorrect but it is something worth noting.


In [ ]:
# Investigate rows with Experience less than 0.
df[df['Experience'] < 0]

We have 52 rows out of 500(about 1% of data) with neagtive experience The negative values are between -1 and -3. I have a hypothesis that these might be typos in data-entry and the negative values should be positive. When looking at these rows, we can see that the ages of these customers are between 23-29 Most people who are at this age have a small amount of experience so 1-3 years of experience for this age range is realistic. Also if we look at their income, we see that there are values that are above 100,000 thousand dollars. To have an income that high for no years of experience would be highly unlikely so that is why I have chosen to convert these negative experience into the absolute values. We will do this after spliting to train and test.

## Univariate Analysis

In [ ]:
continous_columns = ['Age','Experience','Income','CCAvg','Mortgage']
fig,axes = plt.subplots(nrows = 3, ncols = 2, figsize = (10,20))
axes_flat = axes.flatten()
for i, col_name in enumerate(continous_columns):
    sns.histplot(x = df[col_name], kde = True, ax = axes_flat[i])
    axes_flat[i].set_title(f'Histogram for {col_name}')
axes_flat[5].set_visible(False)
plt.tight_layout()
plt.show()

From these plots:
- Age and Experience are not skewed(although not bell shaped). They are both similary distributed.
- Income is positively skewed to the right with majority of people with smaller incomes and then we have a few people on the upper end income.
- CCAvg is also positively skewed to the right and similar to income. One important note is that there is a high proportion of customers who have 0 as CCAvg. This may indicate they don't have a credit card.
- Mortgage is also heavily skewed to the right. Majority of the customers in our dataset do not have a mortgage. This couuld be because they are too young and have no bought a house or maybe they have finsihed of paying a mortgage.

In [ ]:
categorical_columns = ['Family','Education','Personal_Loan','Securities_Account','CD_Account','Online','CreditCard']
fig, axes = plt.subplots(nrows = 4, ncols = 2, figsize = (10,20))
axes_flatten = axes.flatten()
for i,col_name in enumerate(categorical_columns):
    percentage_split_categorical_fields =  (df[col_name].value_counts(normalize = True) * 100).reset_index()
    percentage_split_categorical_fields.columns = [col_name, 'Percentage']
    sns.barplot(x = col_name,y = 'Percentage',data = percentage_split_categorical_fields,ax = axes_flatten[i])
    axes_flatten[i].set_title(f'Countplot of {col_name}')
    axes_flatten[i].set_ylabel('% in each category')
axes_flatten[7].set_visible(False)
plt.tight_layout()
plt.show()


From these plots:
- Family, Education, OnlineAccount are quite balanced.
- Credit Card is slightly unbalanced.
- CD_Account, Securities_Account and PersonalLoan are heavily unbalanced

## Bivariate Analysis

In [ ]:
continous_columns = ['Age','Experience','Income','Mortgage']
fig,axes = plt.subplots(nrows = 2, ncols = 2,figsize = (10,20))
axes_flatten = axes.flatten()
for i, col_name in enumerate(continous_columns):
    sns.regplot(x = col_name, y = 'CCAvg',data = df,ax = axes_flatten[i],scatter_kws = {'color':'blue'},line_kws = {'color':'red'})
    axes_flatten[i].set_title(f'Scatterplot of {col_name}')
    axes_flatten[i].set_xlabel(col_name)
plt.tight_layout()
plt.show()

From these plots we notice:
- The regression line is flat for Age, Experience which shows by themselves there is no correlation with CCAvg. This is also indicated by the scatterplots not forming a pattern.
- For income there is a clear trend that indicates as income increases the CCAvg increases. This is something I would expect since having a higher income would increase purchasing power so will be able to spend more on the credit card and may aslo have a higher limit.
- For Mortgage it is showing a positive correlation. Not as strong as Income. Although this correlation line is dependent on the fact that majority of customers have no mortgage.

In [ ]:
discrete_columns = ['Family','Education','Personal_Loan','Securities_Account','CD_Account','Online','CreditCard']
fig,axes = plt.subplots(nrows = 7, ncols = 1, figsize = (10,50))
axes_flatten = axes.flatten()
for i,col_name in enumerate(discrete_columns):
     sns.boxplot(x = df[col_name].astype(str),y = df['CCAvg'],ax = axes_flatten[i])
#sns.boxplot(x = df['Family'].astype(str),y = df['CCAvg'])

From these boxplots we can say:
- Family,Education, Securities Account, Online do not appear to have a strong relationship with CCAvg.
- PersonalLoan, people who accept a loan tend to have a higher CCAvg. This may be an interesting observation to note when moving onto predicting if a customer will accept a loan.
- CD Account there is a fairly strong relationship in that customers who have a CD Account are likely to spend more. This is something I would expect since if customers are able to put away money into a fixed deposit and not touch the money, they may be more financially stable and this can lead to them being able to spend more on credit card.

In [ ]:
continous_columns = ['CCAvg','Age','Experience','Income','Mortgage',]
correlation_matrix = df[continous_columns].corr()
sns.heatmap(correlation_matrix,
            annot = True,
            cmap = 'coolwarm',
            square = True)
plt.show()

In [ ]:
continous_columns = ['Age','Experience','Income','Mortgage','CCAvg']
sns.pairplot(df[continous_columns])
plt.show()

From the heatmap, main observations:
- Income has a strong positive linear relationship with CCAvg.
- Age and Experience had a very strong linear correlation so when it comes to modelling will need to consider that so we don't end up with multicolinearity.
- From the pairplot, we see for majority of the data, it is scattered around and there are no obvious trends or patterns we can see from this at this. This is both for linear trends and other types of trends.

## Data Pre-Processing

### Train-test Split

In [ ]:
feature_columns = ['Age','Experience','Income','Family','Education','Mortgage','Securities_Account','CD_Account','Online','CreditCard']
X = df[feature_columns]
y_reg = df['CCAvg']
y_clf = df['Personal_Loan']
X_train,X_test,y_reg_train,y_reg_test,y_clf_train,y_clf_test = train_test_split(X,y_reg,y_clf,test_size = 0.2,random_state = 1,stratify=y_clf)

In [ ]:
continous_features = ['Age','Experience','Income','Mortgage']
X_train[continous_features].describe().T

In [ ]:
fig,axes = plt.subplots(2,2,figsize = (12,12))
axes_flatten = axes.flatten()
for i, col_name in enumerate(continous_features):
    sns.boxplot(y = X_train[col_name],ax = axes_flatten[i])
    axes_flatten[i].set_title(f'Boxplot of {col_name}')
plt.tight_layout()
plt.show()

In [ ]:
print("Target variable summary statistics:")
display(y_reg_train.describe())

In [ ]:
plt.figure(figsize=(5, 4))
sns.boxplot(y=y_reg_train)
plt.title("CCAvg (Training Target)")
plt.show()

In [ ]:
log_features = ['Income','Mortgage']
numeric_features = ['Age']
ordinal_features = ['Education','Family']
binary_features = ['Securities_Account','CD_Account','Online','CreditCard']
absolute_features = ['Experience']

In [ ]:
numeric_pipeline = Pipeline([
    ('scaler',StandardScaler())
])
log_pipeline = Pipeline([
    ('log',FunctionTransformer(np.log1p,validate = False)),
    ('scaler',StandardScaler())
])
experience_pipeline = Pipeline([
    ('absolute',FunctionTransformer(np.abs,validate = False)),
    ('scaler',StandardScaler())
])

preprocessor = ColumnTransformer([
    ('age',numeric_pipeline,numeric_features),
    ('exp',experience_pipeline,absolute_features),
    ('log',log_pipeline,log_features),
    ('ord','passthrough',ordinal_features),
    ('bin','passthrough',binary_features)
])



In [ ]:
preprocessor.fit(X_train)

In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_processed,y_reg_train)

In [ ]:
pred_reg = lin_reg.predict(X_test_processed)

In [ ]:
def adj_r2_score(predictors, r2):
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))

def model_evaluation(mae,mape,rmse,mse,r2,adjusted_r2):
    evaluation_metrics = {'MSE':mse,'RMSE':rmse,'MAE':mae,'MAPE':mape,'R^2':r2,'Adjusted R^2':adjusted_r2}
    evaluation_metrics = pd.DataFrame(evaluation_metrics,index = [0])
    return evaluation_metrics

In [ ]:
mae = mean_absolute_error(y_reg_test,pred_reg)
mape = mean_absolute_percentage_error(y_reg_test,pred_reg)
rmse = root_mean_squared_error(y_reg_test,pred_reg)
mse = rmse ** 2
r2 = r2_score(y_reg_test,pred_reg)
adjusted_r2 = adj_r2_score(X_train_processed,r2)
regression_model_evaluation = model_evaluation(mae,mape,rmse,mse,r2,adjusted_r2)
regression_model_evaluation

In [ ]:
feature_names = preprocessor.get_feature_names_out()
coef_table = pd.DataFrame({'feature': feature_names, 'coefficient': lin_reg.coef_}).sort_values(
    'coefficient', key=abs, ascending=False).reset_index(drop=True)
coef_table
